In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

C:\Users\thewr\AppData\Local\Temp\ipykernel_3712\2565918290.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\thewr\git\LSTM_giroscopio_acelerometro.git\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [4]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [5]:
df_csv_gyro = combine_csv_data(csv_gyro_dir)

In [6]:
df_csv_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_x','giroscópio_Y','giroscópio_Z']

In [7]:
df_csv_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426757696641,0.314944,-1.022277,-0.3099616;
1,1600,A,90426807196641,0.387382,-0.618541,-0.048971802;
2,1600,A,90426856696641,0.070999,-0.209480,-0.1959783;
3,1600,A,90426906196641,0.037975,0.254976,-0.1565635;
4,1600,A,90426955696641,0.073129,0.719431,-0.0010349044;


In [8]:
df_csv_gyro.shape

(3440342, 6)

In [9]:
df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [10]:
df_csv_gyro.head()

giroscópio_x  giroscópio_Y  \
pessoa_gyro atividade_gyro tempo_gyro                                   
1600        A              90426757696641      0.314944     -1.022277   
                           90426807196641      0.387382     -0.618541   
                           90426856696641      0.070999     -0.209480   
                           90426906196641      0.037975      0.254976   
                           90426955696641      0.073129      0.719431   

                                             giroscópio_Z  
pessoa_gyro atividade_gyro tempo_gyro                      
1600        A              90426757696641     -0.3099616;  
                           90426807196641   -0.048971802;  
                           90426856696641     -0.1959783;  
                           90426906196641     -0.1565635;  
                           90426955696641  -0.0010349044;

In [11]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3440342 entries, (1600, 'A', 90426757696641) to (1650, 'S', 2426184571385262)
Data columns (total 3 columns):
 #   Column        Dtype  
---  ------        -----  
 0   giroscópio_x  float64
 1   giroscópio_Y  float64
 2   giroscópio_Z  object 
dtypes: float64(2), object(1)
memory usage: 253.2+ MB


In [12]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [13]:
df_csv_accel.shape

(3777046, 6)

In [14]:
df_csv_accel.columns = ['pessoa_acelelômetro','atividade_acelerômetro','tempo_acelelômetro','acelelômetro_x','acelelômetro_Y','acelelômetro_Z']

In [15]:
df_csv_accel.head()

,pessoa_acelelômetro,atividade_acelerômetro,tempo_acelelômetro,acelelômetro_x,acelelômetro_Y,acelelômetro_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502;
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316;
2,1600,A,90426807196641,3.253720,-0.191835,6.107758;
3,1600,A,90426856696641,2.801216,-0.155922,5.997625;
4,1600,A,90426906196641,3.770868,-1.051354,7.731027;


In [16]:
df_csv_accel.set_index(["pessoa_acelelômetro","atividade_acelerômetro","tempo_acelelômetro"],inplace=True)

In [17]:
df_csv_accel.head()

acelelômetro_x  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                   
1600                A                      90426708196641            7.091625   
                                           90426757696641            4.972757   
                                           90426807196641            3.253720   
                                           90426856696641            2.801216   
                                           90426906196641            3.770868   

                                                               acelelômetro_Y  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                   
1600                A                      90426708196641           -0.591667   
                                           90426757696641           -0.158317   
                                           90426807196641           -0.191835   
                                           90426856696641           -0.155922   
                                           90426906196641           -1.051354   

                                                              acelelômetro_Z  
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                 
1600                A                      90426708196641          8.195502;  
                                           90426757696641         6.6967316;  
                                           90426807196641          6.107758;  
                                           90426856696641          5.997625;  
                                           90426906196641          7.731027;

In [18]:
# Select rows with matching indices (inner join)
common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1)

In [19]:
result.shape

(3238804, 6)

In [20]:
result.tail()

acelelômetro_x  acelelômetro_Y acelelômetro_Z  \
1650 S 2426184371437252        2.137723       -7.211485      8.065467;   
       2426184421424592        1.486500      -13.302334      5.355231;   
       2426184471411932        4.017171      -13.146711     5.1182046;   
       2426184521399272        4.412214      -15.318253     3.3704374;   
       2426184571385262        4.426579      -13.012636     2.5755625;   

                         giroscópio_x  giroscópio_Y giroscópio_Z  
1650 S 2426184371437252     -2.856065      2.057101  0.85393304;  
       2426184421424592     -2.911458      1.817417   1.2811041;  
       2426184471411932     -3.015854      1.550036   2.0576818;  
       2426184521399272     -3.218254      1.595842   3.0185504;  
       2426184571385262     -2.455525      0.866136   3.3541086;

In [21]:
result.reset_index(inplace=True)

In [22]:
result.tail()

,level_0,level_1,level_2,acelelômetro_x,acelelômetro_Y,acelelômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
3238799,1650,S,2426184371437252,2.137723,-7.211485,8.065467;,-2.856065,2.057101,0.85393304;
3238800,1650,S,2426184421424592,1.486500,-13.302334,5.355231;,-2.911458,1.817417,1.2811041;
3238801,1650,S,2426184471411932,4.017171,-13.146711,5.1182046;,-3.015854,1.550036,2.0576818;
3238802,1650,S,2426184521399272,4.412214,-15.318253,3.3704374;,-3.218254,1.595842,3.0185504;
3238803,1650,S,2426184571385262,4.426579,-13.012636,2.5755625;,-2.455525,0.866136,3.3541086;


In [23]:
result.columns = ['pessoa_x','atividade','tempo','acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'] 

In [24]:
result.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426757696641,4.972757,-0.158317,6.6967316;,0.314944,-1.022277,-0.3099616;
1,1600,A,90426807196641,3.253720,-0.191835,6.107758;,0.387382,-0.618541,-0.048971802;
2,1600,A,90426856696641,2.801216,-0.155922,5.997625;,0.070999,-0.209480,-0.1959783;
3,1600,A,90426906196641,3.770868,-1.051354,7.731027;,0.037975,0.254976,-0.1565635;
4,1600,A,90426955696641,4.661511,0.169689,9.684695;,0.073129,0.719431,-0.0010349044;


In [25]:
result.atividade.value_counts().sort_index()

atividade
A    178943
B    174412
C    176058
D    185386
E    188118
F    170089
G    178819
H    178156
I    179133
J    181507
K    187798
L    178550
M    178223
O    176805
P    181466
Q    185565
R    178615
S    181161
Name: count, dtype: int64

In [26]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [27]:
result["atividade"] = result["atividade"].replace(to_replace=mapping_dict)

In [28]:
result.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426757696641,4.972757,-0.158317,6.6967316;,0.314944,-1.022277,-0.3099616;
1,1600,0,90426807196641,3.253720,-0.191835,6.107758;,0.387382,-0.618541,-0.048971802;
2,1600,0,90426856696641,2.801216,-0.155922,5.997625;,0.070999,-0.209480,-0.1959783;
3,1600,0,90426906196641,3.770868,-1.051354,7.731027;,0.037975,0.254976,-0.1565635;
4,1600,0,90426955696641,4.661511,0.169689,9.684695;,0.073129,0.719431,-0.0010349044;


In [29]:
result.tail()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
3238799,1650,17,2426184371437252,2.137723,-7.211485,8.065467;,-2.856065,2.057101,0.85393304;
3238800,1650,17,2426184421424592,1.486500,-13.302334,5.355231;,-2.911458,1.817417,1.2811041;
3238801,1650,17,2426184471411932,4.017171,-13.146711,5.1182046;,-3.015854,1.550036,2.0576818;
3238802,1650,17,2426184521399272,4.412214,-15.318253,3.3704374;,-3.218254,1.595842,3.0185504;
3238803,1650,17,2426184571385262,4.426579,-13.012636,2.5755625;,-2.455525,0.866136,3.3541086;


In [30]:
cols_to_check = ['acelerômetro_Z','giroscópio_Z']

In [31]:
result[cols_to_check] = result[cols_to_check].replace({';':''}, regex=True)

In [32]:
result.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426757696641,4.972757,-0.158317,6.6967316,0.314944,-1.022277,-0.3099616
1,1600,0,90426807196641,3.253720,-0.191835,6.107758,0.387382,-0.618541,-0.048971802
2,1600,0,90426856696641,2.801216,-0.155922,5.997625,0.070999,-0.209480,-0.1959783
3,1600,0,90426906196641,3.770868,-1.051354,7.731027,0.037975,0.254976,-0.1565635
4,1600,0,90426955696641,4.661511,0.169689,9.684695,0.073129,0.719431,-0.0010349044


In [33]:
result.to_csv('activity_recognition_data_watch_join_multi_index.csv', index=False)